![electronic_medical_records](electronic_medical_records.png)

Medical professionals often summarize patient encounters in transcripts written in natural language, which include details about symptoms, diagnosis, and treatments. These transcripts can be used for other medical documentation, such as for insurance purposes, but as they are densely packed with medical information, extracting the key data accurately can be challenging.  

You and your team at Lakeside Healthcare Network have decided to leverage the OpenAI API to automatically extract medical information from these transcripts and automate the matching with the appropriate ICD-10 codes. ICD-10 codes are a standardized system used worldwide for diagnosing and billing purposes, such as insurance claims processing.

## The Data
The dataset contains anonymized medical transcriptions categorized by specialty.

## transcriptions.csv
| Column     | Description              |
|------------|--------------------------|
| `"medical_specialty"` | The medical specialty associated with each transcription.  |
| `"transcription"` | Detailed medical transcription texts, with insights into the medical case. |

In [14]:
# Import the necessary libraries
import pandas as pd
from openai import OpenAI
import json

In [15]:
# Initialize the client (the key is preset in the DataCamp environment)
client = OpenAI()

In [16]:
# Load the data
df = pd.read_csv("data/transcriptions.csv")
df.head()
print(df.columns)

Index(['medical_specialty', 'transcription'], dtype='object')


In [17]:
## Start coding here, use as many cells as you need

# Function definition for extracting patient data
extract_patient_data_tool = [{
    "type": "function",
    "function": {
        "name": "extract_patient_info",
        "description": "Extract the patient's age and recommended treatment/procedure from the transcription.",
        "parameters": {
            "type": "object",
            "properties": {
                "age": {"type": "string", "description": "The age of the patient."},
                "treatment": {"type": "string", "description": "The recommended treatment or procedure."}
            },
            "required": ["age", "treatment"]
        }
    }
}]

# Function definition for extracting the ICD-10 code
extract_icd_code_tool = [{
    "type": "function",
    "function": {
        "name": "extract_icd_code",
        "description": "Extract the most likely ICD-10 code for the given medical condition/treatment.",
        "parameters": {
            "type": "object",
            "properties": {
                "icd_code": {"type": "string", "description": "The matched ICD-10 code."}
            },
            "required": ["icd_code"]
        }
    }
}]

In [18]:
structured_data = []

# Iterate through the dataset (using .head(5) for testing, remove to run all)
# Note: The API will take a while if you run the whole dataset!
for index, row in df.head().iterrows(): 
    transcription = row['transcription']
    specialty = row['medical_specialty']
    
    # 1. Extract Age and Treatment
    response_1 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a medical data extractor. Extract the patient's age and the recommended treatment."},
            {"role": "user", "content": transcription}
        ],
        tools=extract_patient_data_tool,
        tool_choice={"type": "function", "function": {"name": "extract_patient_info"}}
    )
    
    # Parse the output
    arguments_1 = json.loads(response_1.choices[0].message.tool_calls[0].function.arguments)
    age = arguments_1.get("age", "Unknown")
    treatment = arguments_1.get("treatment", "Unknown")
    
    # 2. Extract ICD Code based on the treatment/specialty
    prompt_for_icd = f"Medical Specialty: {specialty}\nTreatment: {treatment}"
    response_2 = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an expert medical coder. Assign the most appropriate ICD-10 code based on the specialty and treatment."},
            {"role": "user", "content": prompt_for_icd}
        ],
        tools=extract_icd_code_tool,
        tool_choice={"type": "function", "function": {"name": "extract_icd_code"}}
    )
    
    # Parse the ICD output
    arguments_2 = json.loads(response_2.choices[0].message.tool_calls[0].function.arguments)
    icd_code = arguments_2.get("icd_code", "Unknown")
    
    # Append to our list
    structured_data.append({
        "age": age,
        "medical_specialty": specialty,
        "treatment": treatment,
        "icd_code": icd_code
    })

In [19]:
# Convert the list of dictionaries to a pandas DataFrame
df_structured = pd.DataFrame(structured_data)

# Display the result
df_structured

,age,medical_specialty,treatment,icd_code
0,23,Allergy / Immunology,"Try Zyrtec instead of Allegra, or use loratadi...",J30.9
1,41,Orthopedic,operative fixation for Achilles tendon rupture,S86.011A
2,30,Bariatrics,Laparoscopic antecolic antegastric Roux-en-Y g...,Z68.41
3,50,Cardiovascular / Pulmonary,Neck exploration; tracheostomy; urgent flexibl...,J95.851
4,66,Urology,Self-catheterization in the event of urinary r...,R32
